<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 9B: *Fire Spread Feature Ablation*
##### Version Number: 4.0
---
### Contents
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_spread = pd.read_csv('../data/processed/X_spread.csv')
y_spread = pd.read_csv('../data/processed/y_spread.csv').squeeze()  # Load as Series
details_spread = pd.read_csv('../data/processed/details_spread.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/spread_best_strategy.csv')

with open('../data/processed/model_parameters_spread.json', 'r') as f:
    model_parameters = json.load(f)
    
with open('../data/processed/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_spread,y_spread], axis=1)
subset = subset_df(reform, 'Target_Spread', 500)

y = subset['Target_Spread']
X = subset.drop(columns='Target_Spread')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
spread_xgb = xgb.XGBClassifier(**XGB_parameters)
spread_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 20,
 'min_samples_split': 2,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 5,
 'n_estimators': 200,
 'max_depth': 6,
 'learning_rate': 0.2,
 'verbosity': 0}

In [7]:
models = {
    "XGB": spread_xgb, 
    "RF": spread_rf,
}

## SHAP

In [8]:
xgb_top = get_shap(spread_xgb, X_xgb, y_xgb)
rf_top = get_shap(spread_rf, X_rf, y_rf,)

 99%|===================| 2483/2500 [01:52<00:00]        

In [9]:
weather_features = []

drop = ['fire_count','fire_count 3 Day Sum','fire_count 7 Day Sum','fire_count 30 Day Sum',
       'road_density_x_forest_percent','power_line_density_x_log_total_housing']
weather_sets = ['Water Demand','Water Supply','Water Supply Indexes','Fire Danger','Interactions','Wind Slope',
                              'Lag 3 Day Features','Lag 7 Day Features','Lag 30 Day Features']

for weather_set in weather_sets:
    for feature in feature_sets[weather_set]:
        if feature not in drop:
            weather_features.append(feature)

In [10]:
top_xgb_weather = xgb_top.loc[xgb_top['Feature'].isin(weather_features)]
top_xgb_weather[:5]

,Feature,SHAP Importance
1,1000-hour Dead Fuel Moisture 30 Day Median,0.427130
4,Burning Index 30 Day Mean,0.335388
5,Actual Evapotranspiration 3 Day Sum,0.306242
6,SPEI 180-Day,0.286035
7,Palmer Drought Severity Index,0.281776


In [11]:
top_rf_weather = rf_top.loc[rf_top['Feature'].isin(weather_features)]
top_rf_weather[:5]

,Feature,SHAP Importance
5,1000-hour Dead Fuel Moisture 30 Day Median,0.010451
7,Palmer Drought Severity Index,0.008153
10,SPEI 90-Day,0.007383
11,Energy Release Component 30 Day Mean,0.007039
19,SPEI 180-Day,0.005522


In [12]:
common_ranked = (
    rf_top[:20][['Feature', 'SHAP Importance']]
    .rename(columns={'SHAP Importance': 'RF_Importance'})
    .merge(
        xgb_top[:20][['Feature', 'SHAP Importance']]
        .rename(columns={'SHAP Importance': 'XGB_Importance'}),
        on='Feature',
        how='inner'
    )
    .sort_values('XGB_Importance', ascending=False)
    .reset_index(drop=True)
)

In [22]:
xgb_top[:10]

,Feature,SHAP Importance
0,slope_mean,0.756061
1,1000-hour Dead Fuel Moisture 30 Day Median,0.427130
2,road_density_x_forest_percent,0.375205
3,forest_percent,0.345747
4,Burning Index 30 Day Mean,0.335388
5,Actual Evapotranspiration 3 Day Sum,0.306242
6,SPEI 180-Day,0.286035
7,Palmer Drought Severity Index,0.281776
8,slope_max,0.277333
9,SPEI 90-Day,0.254772


In [23]:
rf_top[:10]

,Feature,SHAP Importance
0,forest_percent,0.013450
1,slope_mean,0.012009
2,dominant_province_description_Sierran Steppe-M...,0.011833
3,slope_max,0.011537
4,road_density_x_forest_percent,0.011261
5,1000-hour Dead Fuel Moisture 30 Day Median,0.010451
6,slope_range,0.008804
7,Palmer Drought Severity Index,0.008153
8,elevation_range,0.007707
9,days_since_last_fire,0.007503


In [16]:
common_ranked

,Feature,RF_Importance,XGB_Importance
0,slope_mean,0.012009,0.756061
1,1000-hour Dead Fuel Moisture 30 Day Median,0.010451,0.427130
2,road_density_x_forest_percent,0.011261,0.375205
3,forest_percent,0.013450,0.345747
4,SPEI 180-Day,0.005522,0.286035
5,Palmer Drought Severity Index,0.008153,0.281776
6,slope_max,0.011537,0.277333
7,SPEI 90-Day,0.007383,0.254772
8,days_since_last_fire,0.007503,0.237612
9,power_line_meters,0.005566,0.227161


## Set Ablation

In [17]:
Ablation_single_column = ablation(models, X, y, feature_sets, best_strategy, single_set = False)

Testing XGB: Water Demand
Testing RF: Water Demand
Testing XGB: Water Supply
Testing RF: Water Supply
Testing XGB: Water Supply Indexes
Testing RF: Water Supply Indexes
Testing XGB: Fire Danger
Testing RF: Fire Danger
Testing XGB: Social
Testing RF: Social
Testing XGB: Infrastructure
Testing RF: Infrastructure
Testing XGB: Elevation
Testing RF: Elevation
Testing XGB: WUI
Testing RF: WUI
Testing XGB: Ecoregion
Testing RF: Ecoregion
Testing XGB: Land Cover
Testing RF: Land Cover
Testing XGB: Interactions
Testing RF: Interactions
Testing XGB: Wind Slope
Testing RF: Wind Slope
Testing XGB: Others
Testing RF: Others
Testing XGB: Coded Ecoregions
Testing RF: Coded Ecoregions
Testing XGB: Coded Seasons
Testing RF: Coded Seasons
Testing XGB: Spatial
Testing RF: Spatial
Testing XGB: Lag 3 Day Features
Testing RF: Lag 3 Day Features
Testing XGB: Lag 7 Day Features
Testing RF: Lag 7 Day Features
Testing XGB: Lag 30 Day Features
Testing RF: Lag 30 Day Features


In [18]:
Ablation_single_column

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.767755,0.764700,0.656250,0.000000
1,Water Demand,XGB,0.767215,0.764891,0.656250,-0.025075
2,Water Supply,XGB,0.772228,0.769106,0.677083,-0.576224
3,Water Supply Indexes,XGB,0.752899,0.748766,0.687500,2.083640
4,Fire Danger,XGB,0.789532,0.788176,0.697917,-3.069942
5,Social,XGB,0.787584,0.784761,0.697917,-2.623459
6,Infrastructure,XGB,0.771334,0.768169,0.666667,-0.453650
7,Elevation,XGB,0.752254,0.749119,0.656250,2.037431
8,WUI,XGB,0.767753,0.764941,0.656250,-0.031588
9,Ecoregion,XGB,0.768454,0.765420,0.666667,-0.094137


In [19]:
# Assume your table is in a DataFrame called df
# Filter out the full runs
full_model = Ablation_single_column[Ablation_single_column['Test Set'] == 'Full'][['Model', 'Macro F1']].rename(columns={'Macro F1': 'Full Macro F1'})

# Merge back to the main dataframe on 'Model'
pivot_ablation = Ablation_single_column.merge(full_model, on='Model', how='left')

# Drop columns you don’t need for now
pivot_ablation = pivot_ablation.drop(columns=['Weighted F1', 'High Risk Recall'])

largest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] > 1]
smallest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] < 0]

In [20]:
largest_drops

,Test Set,Model,Macro F1,Macro F1 Percent Difference,Full Macro F1
3,Water Supply Indexes,XGB,0.748766,2.083640,0.764700
7,Elevation,XGB,0.749119,2.037431,0.764700
19,Lag 30 Day Features,XGB,0.756756,1.038766,0.764700
23,Water Supply Indexes,RF,0.756072,2.034762,0.771776
24,Fire Danger,RF,0.759022,1.652490,0.771776
27,Elevation,RF,0.751536,2.622570,0.771776
36,Spatial,RF,0.757471,1.853559,0.771776


In [21]:
smallest_drops

,Test Set,Model,Macro F1,Macro F1 Percent Difference,Full Macro F1
1,Water Demand,XGB,0.764891,-0.025075,0.764700
2,Water Supply,XGB,0.769106,-0.576224,0.764700
4,Fire Danger,XGB,0.788176,-3.069942,0.764700
5,Social,XGB,0.784761,-2.623459,0.764700
6,Infrastructure,XGB,0.768169,-0.453650,0.764700
8,WUI,XGB,0.764941,-0.031588,0.764700
9,Ecoregion,XGB,0.765420,-0.094137,0.764700
10,Land Cover,XGB,0.768866,-0.544813,0.764700
11,Interactions,XGB,0.771783,-0.926295,0.764700
12,Wind Slope,XGB,0.776983,-1.606304,0.764700
